# 02 · พยากรณ์ PM2.5 ล่วงหน้า 1–6 ชั่วโมง (หลักฐานชุด "เมื่อไร")

- เป้าหมาย: ค่า PM2.5 สถานีขอนแก่นที่ t+h
- แบ่งข้อมูลตามเวลา (ห้ามสุ่ม) ช่วงท้าย 20% เป็นชุดทดสอบ
- เทียบกับ persistence (ค่าปัจจุบัน = ค่าอนาคต) ซึ่งเป็นเกณฑ์ขั้นต่ำที่ต้องชนะ
- Backtest การแจ้งเตือนที่เกณฑ์ 37.5 µg/m³: hit rate, false alarm ratio
- Ablation: เพิ่ม feature ก๊าซ (สถานีมหาสารคาม + CAMS) แล้วดีขึ้นหรือไม่
- TODO: SARIMA, LSTM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# หา root ของ repo (โฟลเดอร์ที่มี data/) ไม่ว่าจะเปิด notebook จากที่ไหน
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(PROC / "hourly_khonkaen.csv", parse_dates=["time"], index_col="time").asfreq("h")
TARGET = "station_pm25"
THRESHOLD = 37.5
HORIZONS = [1, 3, 6]
MET = [c for c in ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m",
                   "precipitation", "surface_pressure"] if c in df]
GAS = [c for c in df if c.startswith(("mk_", "cams_")) and "pm" not in c]   # ก๊าซเท่านั้น ไม่รวมฝุ่น
print(len(df), "ชั่วโมง | met:", MET, "| gas:", GAS)

In [ ]:
def make_features(d, cols_extra):
    X = pd.DataFrame(index=d.index)
    y = d[TARGET]
    for L in [0, 1, 2, 3, 6, 12, 24]:
        X[f"lag{L}"] = y.shift(L)
    for w in [3, 6, 24]:
        X[f"roll{w}"] = y.rolling(w, min_periods=1).mean()
    X["hour"], X["dow"], X["month"] = d.index.hour, d.index.dayofweek, d.index.month
    if "wind_direction_10m" in d:
        rad = np.deg2rad(d["wind_direction_10m"])
        X["wind_u"], X["wind_v"] = -d.get("wind_speed_10m", 1) * np.sin(rad), -d.get("wind_speed_10m", 1) * np.cos(rad)
    for c in cols_extra:
        X[c] = d[c]
        X[c + "_lag3"] = d[c].shift(3)
    return X

try:
    from xgboost import XGBRegressor
    def model(): return XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8)
except ImportError:                                   # ถ้ายังไม่ได้ติดตั้ง xgboost
    from sklearn.ensemble import HistGradientBoostingRegressor
    def model(): return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05)

def alarm_scores(y_true, y_pred, thr=THRESHOLD):
    a, p = y_true > thr, y_pred > thr
    hit = (a & p).sum() / max(a.sum(), 1)            # POD: เกินจริงแล้วเตือนทัน
    far = (~a & p).sum() / max(p.sum(), 1)           # เตือนแต่ไม่เกินจริง
    return hit, far

In [ ]:
results = []
for extra_name, extra in [("base", MET), ("base+gas", MET + GAS)]:
    X_all = make_features(df, extra)
    for h in HORIZONS:
        y = df[TARGET].shift(-h).rename("y")
        data = X_all.join(y).dropna(subset=["y", "lag0"])
        cut = data.index[int(len(data) * 0.8)]
        tr, te = data[data.index < cut], data[data.index >= cut]
        m = model().fit(tr.drop(columns="y"), tr["y"])
        pred = pd.Series(m.predict(te.drop(columns="y")), index=te.index)
        for name, p in [("persistence", te["lag0"]), (extra_name, pred)]:
            if name == "persistence" and extra_name != "base":
                continue
            hit, far = alarm_scores(te["y"], p)
            results.append({"model": name, "horizon_h": h,
                            "MAE": (p - te["y"]).abs().mean(),
                            "RMSE": np.sqrt(((p - te["y"]) ** 2).mean()),
                            "hit_rate": hit, "false_alarm": far})
res = pd.DataFrame(results).round(3)
res.to_csv(PROC / "forecast_results.csv", index=False)
res.pivot_table(index="model", columns="horizon_h", values="MAE")

In [ ]:
res